# Stage 09 / Lecture 60: Model Execution Modes (Invoke, Streaming, & Parallel Batching)

This production reference notebook benchmarks and validates the three core `Runnable` execution interfaces in **LangChain v1+** using `ChatGroq`:
1. **Standard Blocking Inference (`model.invoke`)**: Full-sequence waiting baseline.
2. **Progressive Token Streaming (`model.stream`)**: Measuring Time-to-First-Token (`TTFT`), yielding `AIMessageChunk` objects, and assembling progressive outputs cleanly.
3. **Concurrency-Gated Parallel Batching (`model.batch`)**: Executing multiple independent prompts concurrently while protecting provider rate limits via `max_concurrency`.


In [1]:
from typing import List, Dict, Any, Tuple
import os
import time
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_core.messages import AIMessage, AIMessageChunk

load_dotenv()

if not os.getenv("GROQ_API_KEY"):
    raise EnvironmentError("GROQ_API_KEY must be configured in .env to run execution benchmarks.")

model = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)
print("ChatGroq model initialized successfully for execution benchmarks.")


ChatGroq model initialized successfully for execution benchmarks.


### 1. Standard Blocking Invocation (`model.invoke()`)

In standard blocking execution, the caller waits until 100% of output tokens are generated on the provider inference cluster before receiving a single `AIMessage` payload.


In [2]:
prompt_single = "Explain in 2 short sentences why parrots have colorful feathers."

t_start_invoke = time.perf_counter()
invoke_response = model.invoke(prompt_single)
t_total_invoke = time.perf_counter() - t_start_invoke

assert isinstance(invoke_response, AIMessage)
assert len(invoke_response.content.strip()) > 0

print(f"Invoke Total Wait Latency (TTFT == Total): {t_total_invoke:.3f}s")
print(f"Response Content: {invoke_response.content}")


Invoke Total Wait Latency (TTFT == Total): 1.101s
Response Content: Parrots display bright feathers to attract mates and signal good health. The colors also help them communicate and recognize each other in the wild.


### 2. Progressive Token Streaming (`model.stream()`) and TTFT Verification

`model.stream()` returns an iterator yielding `AIMessageChunk` deltas in real time. Using `chunk.text` and `flush=True` allows UI layers to display tokens milliseconds after request dispatch.


In [3]:
t_start_stream = time.perf_counter()
ttft_stream: float = 0.0
chunks: List[AIMessageChunk] = []

print("Streaming Output Preview:")
for idx, chunk in enumerate(model.stream(prompt_single)):
    if idx == 0:
        ttft_stream = time.perf_counter() - t_start_stream
    chunks.append(chunk)
    print(chunk.text, end="", flush=True)

t_total_stream = time.perf_counter() - t_start_stream
assembled_text = "".join(c.text for c in chunks)

assert len(chunks) > 5, f"Expected multiple streamed token chunks, got {len(chunks)}"
assert all(isinstance(c, AIMessageChunk) for c in chunks)
assert len(assembled_text.strip()) > 0

print(f"\n\nStream Chunks Count: {len(chunks)}")
print(f"Stream Time-to-First-Token (TTFT): {ttft_stream:.3f}s")
print(f"Stream Total Latency: {t_total_stream:.3f}s")


Streaming Output Preview:


Par

rots

 display

 bright

 feathers

... (32 outputs left out)

 forest

 habitats

.



Stream Chunks Count: 184
Stream Time-to-First-Token (TTFT): 0.206s
Stream Total Latency: 0.600s


### 3. Parallel Batch Processing (`model.batch()`) with `max_concurrency` Control

For offline pipelines, synthetic evaluations, or multi-document extraction, `model.batch()` dispatches independent requests concurrently. Setting `config={"max_concurrency": N}` prevents triggering HTTP 429 rate limits.


In [4]:
batch_prompts: List[str] = [
    "Why do parrots have colorful feathers? Answer in 1 concise sentence.",
    "How do airplanes generate lift to fly? Answer in 1 concise sentence.",
    "What is quantum computing? Answer in 1 concise sentence."
]

# Execute concurrently with bounded max_concurrency=3
t_start_batch = time.perf_counter()
batch_responses: List[AIMessage] = model.batch(
    batch_prompts,
    config={"max_concurrency": 3}
)
t_total_batch = time.perf_counter() - t_start_batch

assert len(batch_responses) == len(batch_prompts)
assert all(isinstance(r, AIMessage) and len(r.content.strip()) > 0 for r in batch_responses)

print(f"Processed {len(batch_prompts)} independent prompts concurrently in {t_total_batch:.3f}s (max_concurrency=3):")
for idx, (q, resp) in enumerate(zip(batch_prompts, batch_responses), 1):
    print(f"  [{idx}] Q: {q}")
    print(f"      A: {resp.content.strip()}")

print("\nAll Stage 09 / Lecture 60 streaming and batch invariants verified successfully.")


Processed 3 independent prompts concurrently in 2.142s (max_concurrency=3):
  [1] Q: Why do parrots have colorful feathers? Answer in 1 concise sentence.
      A: Parrots’ bright feathers evolved through sexual selection to attract mates and signal species identity, while also helping them blend into forest habitats.
  [2] Q: How do airplanes generate lift to fly? Answer in 1 concise sentence.
      A: Airplanes generate lift by shaping their wings so that air moves faster over the curved upper surface, creating lower pressure above and higher pressure below, which pushes the wing upward.
  [3] Q: What is quantum computing? Answer in 1 concise sentence.
      A: Quantum computing harnesses qubits that can occupy superpositions and become entangled, enabling certain calculations to be performed exponentially faster than with classical bits.

All Stage 09 / Lecture 60 streaming and batch invariants verified successfully.
